# Limpeza de dados na prática
**Prof. João Choma — Unicive**

Vamos abrir uma base suja e limpar um problema de cada vez.

**Antes de começar no Colab:** clique na pasta da barra lateral, envie o arquivo
`operacoes.csv` fornecido com a aula e aguarde o envio terminar. Mantenha esse nome.
O arquivo deve aparecer na pasta principal da sessão, junto do local em que o notebook executa.
Se aparecer `FileNotFoundError`, confira o envio e o nome do arquivo.

Depois, execute as células na ordem, usando **Shift+Enter**.
A base tem somente 16 linhas para podermos acompanhar as mudanças.


## 1. Ler a base suja

`pandas` é a biblioteca que usaremos. Chamaremos a tabela de `df`.

`read_csv` abre o arquivo e `head()` mostra as cinco primeiras linhas.


In [1]:
import pandas as pd

df = pd.read_csv("operacoes.csv")
df.head()


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
0,ESC-01,01/08/2026,8,95.0,120,Carlos,Obra A
1,esc-01,02/08/2026,9,110.0,135,Ana,obra a
2,ESC-01,03/08/2026,NaN,100.0,125,Carlos,Obra A
3,ESC 01,04/08/2026,7.5,82.0,110,Ana,Obra A
4,ESC-02,01/08/2026,8,90.0,100,João,Obra B


**O que observar:** A tabela mostra equipamento, data, horas, combustível, produção, operador e obra. Já aparece um valor ausente em horas, normalmente exibido como `NaN`.


## 2. Ver a tabela inteira

Como a base é pequena, podemos olhar todos os registros. Procure espaços nos nomes, números negativos, palavras no lugar de números e datas estranhas.


In [2]:
df


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
0,ESC-01,01/08/2026,8,95.0,120,Carlos,Obra A
1,esc-01,02/08/2026,9,110.0,135,Ana,obra a
2,ESC-01,03/08/2026,NaN,100.0,125,Carlos,Obra A
3,ESC 01,04/08/2026,7.5,82.0,110,Ana,Obra A
4,ESC-02,01/08/2026,8,90.0,100,João,Obra B
5,ESC-02,02/08/2026,-4,50.0,80,João,Obra B
6,ESC-03,01/08/2026,10,150.0,180,Marcos,Obra C
7,ESC-03,02/08/2026,oito,120.0,160,Marcos,Obra C
8,ESC-04,01/08/2026,6,68.0,85,Paulo,Obra B
9,ESC-04,02/08/2026,7,NaN,95,Paulo,Obra B


**O que observar:** Procure `esc-01`, `ESC 01`, `oito`, `-4`, zero horas e `35/08/2026`. Encontrar esses valores é o início do diagnóstico; ainda não vamos corrigir tudo de uma vez.


## 3. Descobrir o tamanho e os tipos

`shape` informa linhas e colunas. `info()` mostra o tipo de cada coluna e quantos valores estão preenchidos.


In [3]:
print("Linhas e colunas:", df.shape)
df.info()


Linhas e colunas: (16, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   equipamento         16 non-null     object 
 1   data                16 non-null     object 
 2   horas_trabalhadas   15 non-null     object 
 3   combustivel_litros  15 non-null     float64
 4   producao_m3         16 non-null     int64  
 5   operador            16 non-null     object 
 6   obra                16 non-null     object 
dtypes: float64(1), int64(1), object(5)
memory usage: 1.0+ KB


**O que observar:** São **16 linhas e 7 colunas**. Horas não está como coluna numérica por causa de `oito`. Datas ainda são texto. Horas e combustível têm 15 valores preenchidos cada. Os nomes dos tipos podem variar entre versões do pandas.


## 4. Contar os valores ausentes

`isna()` identifica valores ausentes. `sum()` conta quantos existem por coluna.


In [4]:
df.isna().sum()


equipamento           0
data                  0
horas_trabalhadas     1
combustivel_litros    1
producao_m3           0
operador              0
obra                  0
dtype: int64

**O que observar:** Há **1 ausência em horas** e **1 em combustível**. `oito` e a data impossível ainda não são ausentes: são textos preenchidos que precisam ser investigados.


## 5. Guardar uma cópia antes de alterar

Vamos manter a tabela original em outra variável. Assim podemos comparar os valores depois. O arquivo CSV original também será preservado.


In [5]:
df_original = df.copy()
print("Cópia guardada:", len(df_original), "linhas")


Cópia guardada: 16 linhas


**O que observar:** `df_original` continuará com 16 linhas. A partir daqui, as alterações serão feitas em `df`. Para recomeçar, execute novamente desde a leitura do arquivo.


## 6. Encontrar linhas repetidas

`duplicated(keep=False)` mostra todas as ocorrências de linhas iguais, inclusive a primeira.


In [6]:
df[df.duplicated(keep=False)]


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
4,ESC-02,01/08/2026,8,90.0,100,João,Obra B
15,ESC-02,01/08/2026,8,90.0,100,João,Obra B


**O que observar:** Os índices **4 e 15** mostram a mesma operação de ESC-02 em 01/08/2026. São duas ocorrências, das quais uma é excedente. Não basta equipamento e data coincidirem: aqui estamos comparando todas as colunas.


## 7. Retirar uma cópia repetida

Para esta demonstração, vamos assumir que essa repetição exata foi causada por uma importação duplicada. `drop_duplicates()` mantém a primeira ocorrência. Fazemos isso antes de converter valores, para não confundir textos diferentes que acabem virando ausentes.


In [7]:
df = df.drop_duplicates().copy()
print("Linhas após retirar a cópia:", len(df))


Linhas após retirar a cópia: 15


**O que observar:** Agora são **15 linhas**. A tabela original continua guardada. Em uma situação real, precisamos confirmar se duas linhas iguais representam o mesmo evento.


## 8. Ver as diferentes escritas dos equipamentos

`unique()` lista os valores diferentes de uma coluna.


In [8]:
df["equipamento"].unique()


array(['ESC-01', 'esc-01', ' ESC-01 ', 'ESC 01', 'ESC-02', 'ESC-03',
       'ESC-04', 'ESC-05'], dtype=object)

**O que observar:** Aparecem **8 grafias**, incluindo `ESC-01`, `esc-01`, ` ESC-01 ` e `ESC 01`. Nesta base, essas quatro grafias representam o mesmo equipamento.


## 9. Padronizar os códigos

Vamos fazer três alterações visíveis: remover espaços externos, passar para maiúsculas e corrigir uma equivalência conhecida. Não vamos trocar espaços por hífens em qualquer texto.


In [9]:
df["equipamento"] = df["equipamento"].str.strip()
df["equipamento"] = df["equipamento"].str.upper()
df["equipamento"] = df["equipamento"].replace({"ESC 01": "ESC-01"})

df["equipamento"].value_counts()


equipamento
ESC-01    5
ESC-02    3
ESC-03    3
ESC-04    2
ESC-05    2
Name: count, dtype: int64

**O que observar:** Agora são **5 códigos**. As frequências são ESC-01: **5**, ESC-02: **3**, ESC-03: **3**, ESC-04: **2** e ESC-05: **2**. A quantidade de linhas permanece 15.


## 10. Padronizar os nomes das obras

Adotaremos os nomes das obras em maiúsculas e sem espaços nas extremidades.


In [10]:
df["obra"] = df["obra"].str.strip()
df["obra"] = df["obra"].str.upper()

df["obra"].value_counts()


obra
OBRA A    7
OBRA B    5
OBRA C    3
Name: count, dtype: int64

**O que observar:** Ficam **3 obras**: OBRA A com **7** operações, OBRA B com **5** e OBRA C com **3**. A padronização reuniu grafias diferentes; não apagou operações.


## 11. Transformar horas em números

`to_numeric` converte os valores. Com `errors="coerce"`, o que não puder ser convertido fica ausente. A conversão não adivinha a medição correta.


In [11]:
df["horas_trabalhadas"] = pd.to_numeric(df["horas_trabalhadas"], errors="coerce")

df[["equipamento", "horas_trabalhadas"]]


,equipamento,horas_trabalhadas
0,ESC-01,8.0
1,ESC-01,9.0
2,ESC-01,NaN
3,ESC-01,7.5
4,ESC-02,8.0
5,ESC-02,-4.0
6,ESC-03,10.0
7,ESC-03,NaN
8,ESC-04,6.0
9,ESC-04,7.0


**O que observar:** `oito` virou ausente. Agora há **2 ausências em horas**: a original e a falha de conversão. **-4 continua -4**: ser número não significa respeitar a regra do domínio.


## 12. Localizar os valores que ficaram ausentes

Colocamos a condição dentro de `df[...]` para mostrar somente as linhas em que horas está ausente.


In [12]:
df[df["horas_trabalhadas"].isna()]


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
2,ESC-01,03/08/2026,NaN,100.0,125,Carlos,OBRA A
7,ESC-03,02/08/2026,NaN,120.0,160,Marcos,OBRA C


**O que observar:** Aparecem os índices **2 e 7**. Podemos consultar `df_original` nesses mesmos índices para descobrir o que havia antes.


## 13. Converter datas

`%d/%m/%Y` significa dia/mês/ano. Datas impossíveis ficam como `NaT`, que representa ausência de data.


In [13]:
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y", errors="coerce")

df[df["data"].isna()]


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
13,ESC-03,NaT,9.0,118.0,155,Marcos,OBRA C


**O que observar:** A linha de índice **13**, com `35/08/2026`, aparece sem data válida. Não sabemos qual dia foi realmente registrado; por isso não vamos inventar uma data.


## 14. Procurar horas negativas

Horas trabalhadas não podem ser negativas neste contexto. Primeiro vamos mostrar o problema, sem alterar nada.


In [14]:
df[df["horas_trabalhadas"] < 0]


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
5,ESC-02,2026-08-02,-4.0,50.0,80,João,OBRA B


**O que observar:** O índice **5** contém **-4 horas**. Trocar automaticamente por 4 seria uma suposição sobre o que a pessoa pretendia registrar.


## 15. Marcar o valor inválido como ausente

Em `.loc[condição, coluna]`, a condição escolhe as linhas e o segundo item escolhe a coluna que será alterada. Usaremos `float("nan")` para representar ausência numérica.


In [15]:
df.loc[df["horas_trabalhadas"] < 0, "horas_trabalhadas"] = float("nan")

df.isna().sum()


equipamento           0
data                  1
horas_trabalhadas     3
combustivel_litros    1
producao_m3           0
operador              0
obra                  0
dtype: int64

**O que observar:** Agora há **3 ausências em horas**, **1 em combustível** e **1 em data**. A quantidade de ausências aumentou porque tornamos problemas explícitos, não porque perdemos valores confiáveis.


## 16. Observar a operação com zero horas

Zero é um número válido, mas não pode ser usado como denominador no cálculo de consumo por hora.


In [16]:
df[df["horas_trabalhadas"] == 0]


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
11,ESC-05,2026-08-02,0.0,40.0,0,Carlos,OBRA A


**O que observar:** O índice **11** tem zero horas e 40 litros. Ele precisa de revisão para a análise por hora. Não alteraremos zero para 1 nem apagaremos esse registro da base tratada.


## 17. Decidir o que fazer com ausências

Para esta aula, vamos preparar uma base de operações completas para analisar consumo por hora.
Escolheremos somente registros com todos os campos preenchidos e horas positivas.

**Não vamos preencher combustível ou horas com zero:** desconhecido não significa zero.
**Não vamos adivinhar a data:** uma data estimada pareceria uma informação real.

Manteremos `df` como base tratada com as 15 operações. A seleção para análise ficará em `df_limpo`.
Os registros fora da seleção não deixam de existir e podem ser aproveitados em outras análises.


## 18. Selecionar registros completos

Aqui `dropna()` é uma escolha explícita: para esta entrega, exigimos todos os sete campos preenchidos. Se outra análise precisar de menos campos, use `dropna(subset=[...])` com os nomes necessários.


In [17]:
df_limpo = df.dropna().copy()
print("Registros completos:", len(df_limpo))


Registros completos: 10


**O que observar:** Restam **10 registros completos**. A operação com zero horas continua presente, pois zero não é ausência.


## 19. Selecionar horas positivas

Agora retiramos da seleção de análise a operação cujo denominador seria zero.


In [18]:
df_limpo = df_limpo[df_limpo["horas_trabalhadas"] > 0].copy()
print("Registros para análise:", len(df_limpo))
df_limpo


Registros para análise: 9


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
0,ESC-01,2026-08-01,8.0,95.0,120,Carlos,OBRA A
1,ESC-01,2026-08-02,9.0,110.0,135,Ana,OBRA A
3,ESC-01,2026-08-04,7.5,82.0,110,Ana,OBRA A
4,ESC-02,2026-08-01,8.0,90.0,100,João,OBRA B
6,ESC-03,2026-08-01,10.0,150.0,180,Marcos,OBRA C
8,ESC-04,2026-08-01,6.0,68.0,85,Paulo,OBRA B
10,ESC-05,2026-08-01,9.0,115.0,140,Ana,OBRA A
12,ESC-02,2026-08-02,8.0,90.0,100,João,OBRA B
14,ESC-01,2026-08-05,8.0,96.0,122,Carlos,OBRA A


**O que observar:** A seleção final tem **9 registros**. Essa é nossa base limpa para a finalidade declarada. As demais operações continuam em `df` e na cópia original.


## 20. Conferir o resultado

Antes de salvar, confira ausências, duplicatas e o menor valor de horas. Essa conferência simples já ajuda a detectar um erro no tratamento.


In [19]:
print("Ausentes por coluna:")
print(df_limpo.isna().sum())
print("Duplicatas:", df_limpo.duplicated().sum())
print("Menor quantidade de horas:", df_limpo["horas_trabalhadas"].min())


Ausentes por coluna:
equipamento           0
data                  0
horas_trabalhadas     0
combustivel_litros    0
producao_m3           0
operador              0
obra                  0
dtype: int64
Duplicatas: 0
Menor quantidade de horas: 6.0


**O que observar:** Todas as colunas têm **zero ausentes**, há **zero duplicatas** e o mínimo é **6 horas**. Estes resultados valem para a base desta aula; não garantem que qualquer outra base esteja correta.


## 21. Comparar antes e depois

Vamos conferir quantas linhas existem em cada etapa, sem somar tabelas que contêm os mesmos registros.


In [20]:
print("Base original:", len(df_original))
print("Base tratada:", len(df))
print("Base selecionada para análise:", len(df_limpo))


Base original: 16
Base tratada: 15
Base selecionada para análise: 9


**O que observar:** **16 originais → 15 após retirar uma cópia → 9 selecionadas**. Seis operações únicas ficaram fora da análise por ausência de informação ou zero horas.


## 22. Ver quais operações ficaram fora da análise

Guardamos os índices originais durante a aula. `isin` verifica quais estão na seleção; `~` inverte a condição. Vamos usar isso apenas agora para conferir o que ficou pendente.


In [21]:
pendentes = df[~df.index.isin(df_limpo.index)].copy()
pendentes


,equipamento,data,horas_trabalhadas,combustivel_litros,producao_m3,operador,obra
2,ESC-01,2026-08-03,NaN,100.0,125,Carlos,OBRA A
5,ESC-02,2026-08-02,NaN,50.0,80,João,OBRA B
7,ESC-03,2026-08-02,NaN,120.0,160,Marcos,OBRA C
9,ESC-04,2026-08-02,7.0,NaN,95,Paulo,OBRA B
11,ESC-05,2026-08-02,0.0,40.0,0,Carlos,OBRA A
13,ESC-03,NaT,9.0,118.0,155,Marcos,OBRA C


**O que observar:** São **6 linhas**, nos índices **2, 5, 7, 9, 11 e 13**. Observe o motivo em cada linha. Esse resultado mostra por que devemos guardar a base tratada, além da seleção limpa.


## 23. Salvar os resultados

`to_csv` grava um arquivo. `index=False` evita adicionar uma coluna com o índice. Usaremos nomes diferentes para preservar `operacoes.csv`.


In [22]:
df_limpo.to_csv("operacoes_limpas.csv", index=False)
df.to_csv("operacoes_tratadas.csv", index=False)
pendentes.to_csv("operacoes_pendentes.csv", index=False)

print("Arquivos salvos.")


Arquivos salvos.


**O que observar:** Os três arquivos aparecem na pasta lateral do Colab. Clique com o botão direito em cada um e escolha baixar. CSV não preserva tipos: ao reabrir para análise, converta a coluna de data novamente.


## 24. Tirar as primeiras informações

A limpeza terminou. Para começar a interpretar, podemos responder somente três perguntas:
quanto os registros selecionados representam em horas, combustível e produção?


In [23]:
print("Horas:", df_limpo["horas_trabalhadas"].sum())
print("Combustível em litros:", df_limpo["combustivel_litros"].sum())
print("Produção em m³:", df_limpo["producao_m3"].sum())


Horas: 73.5
Combustível em litros: 896.0
Produção em m³: 1092


**Resultado esperado:** 73,5 horas, 896 litros e 1.092 m³.
Esses totais representam as **9 operações selecionadas**, e não toda a operação da empresa.

Primeiro entendemos o que cada número representa. Ao final do notebook, a etapa 26
continua a análise com médias, comparação de equipamentos e indicadores por hora.

## 25. Sua vez

Na atividade, envie `operacoes_equipamentos_sujo.csv` pelo painel de arquivos do Colab.
Comece da mesma forma: leia a base, mostre as primeiras linhas e procure os problemas.
A base maior possui 123 linhas e 10 colunas, com casos adicionais. Não espere obter os mesmos números da demonstração.

Use o enunciado `atividades/atividade_limpeza.md`. A solução deve justificar as decisões,
comparar antes/depois e preservar o arquivo bruto.

A célula abaixo é um ponto de partida: retire o `#` da leitura depois de enviar a base da atividade.


In [24]:
# atividade = pd.read_csv("operacoes_equipamentos_sujo.csv")
# atividade.head()


### Roteiro para resolver a atividade

1. Veja o tamanho da tabela, os tipos e os valores ausentes.
2. Procure categorias escritas de maneiras diferentes e linhas repetidas.
3. Converta números e datas, observando o que não converteu.
4. Procure valores inválidos; não apague valores apenas por serem incomuns.
5. Escolha e explique como lidar com cada problema.
6. Confira o resultado e salve com outro nome.

**Para encerrar a aula:** explique uma alteração que você fez e um valor que você preferiu não adivinhar.


## 26. Analisar a base limpa

Agora vamos transformar os dados em informações, com **uma pergunta por vez**.
Usaremos `df_limpo`, a base da demonstração com 9 operações. Não é necessário resolver a atividade
para executar esta etapa: a célula da atividade está comentada e não altera nossos dados.

Já sabemos os totais: **73,5 horas, 896 litros e 1.092 m³**.
Agora queremos entender a duração das operações, a produção dos equipamentos e o consumo por hora.

Os resultados descrevem somente as operações selecionadas. As 6 pendências podem mudar
a comparação se forem recuperadas depois. Por isso, sempre acompanhe uma métrica com a quantidade de registros.


## 26.1. Quanto dura uma operação?

`mean()` calcula a média. `median()` mostra o valor central. `min()` e `max()` mostram os extremos observados. Vamos aplicar os quatro comandos somente à coluna de horas.


In [25]:
print("Média de horas:", round(df_limpo["horas_trabalhadas"].mean(), 2))
print("Mediana de horas:", df_limpo["horas_trabalhadas"].median())
print("Menor duração:", df_limpo["horas_trabalhadas"].min())
print("Maior duração:", df_limpo["horas_trabalhadas"].max())


Média de horas: 8.17
Mediana de horas: 8.0
Menor duração: 6.0
Maior duração: 10.0


**O que observar:** A média é **8,17 horas**, a mediana é **8**, o mínimo é **6** e o máximo é **10**. Podemos dizer que a duração média observada foi 8,17 horas; não que todas as operações duraram esse tempo. **Pergunta:** uma operação de 10 horas é necessariamente um erro? Não: ela está dentro das regras adotadas.


## 26.2. Quantas operações temos de cada equipamento?

`value_counts()` conta as ocorrências de cada código. Antes de comparar desempenho, precisamos saber quantas observações sustentam cada resultado.


In [26]:
df_limpo["equipamento"].value_counts()


equipamento
ESC-01    4
ESC-02    2
ESC-03    1
ESC-04    1
ESC-05    1
Name: count, dtype: int64

**O que observar:** **ESC-01: 4 operações; ESC-02: 2; ESC-03, ESC-04 e ESC-05: 1 cada.** Temos quantidades diferentes de observações. Um único registro pode ser insuficiente para representar o comportamento habitual de uma máquina.


## 26.3. Qual equipamento acumulou mais produção?

`groupby("equipamento")` reúne as operações de cada máquina. Selecionamos a coluna de produção e usamos `sum()` para somar dentro de cada grupo. `sort_values` coloca os maiores totais primeiro.


In [27]:
producao_por_equipamento = df_limpo.groupby("equipamento")["producao_m3"].sum()
producao_por_equipamento.sort_values(ascending=False)


equipamento
ESC-01    487
ESC-02    200
ESC-03    180
ESC-05    140
ESC-04     85
Name: producao_m3, dtype: int64

**O que observar:** **ESC-01: 487 m³; ESC-02: 200 m³; ESC-03: 180 m³; ESC-05: 140 m³; ESC-04: 85 m³.** ESC-01 tem o maior volume registrado, mas também tem mais operações. Maior produção total não comprova maior produtividade por hora.


## 26.4. Quanto cada operação consome e produz por hora?

Vamos criar uma cópia para a análise. Consumo por hora é combustível dividido por horas; produtividade é produção dividida por horas. A limpeza já garantiu horas positivas nesta seleção.


In [28]:
analise = df_limpo.copy()
analise["consumo_l_h"] = analise["combustivel_litros"] / analise["horas_trabalhadas"]
analise["produtividade_m3_h"] = analise["producao_m3"] / analise["horas_trabalhadas"]

analise[["equipamento", "horas_trabalhadas", "consumo_l_h", "produtividade_m3_h"]].round(2)


,equipamento,horas_trabalhadas,consumo_l_h,produtividade_m3_h
0,ESC-01,8.0,11.88,15.00
1,ESC-01,9.0,12.22,15.00
3,ESC-01,7.5,10.93,14.67
4,ESC-02,8.0,11.25,12.50
6,ESC-03,10.0,15.00,18.00
8,ESC-04,6.0,11.33,14.17
10,ESC-05,9.0,12.78,15.56
12,ESC-02,8.0,11.25,12.50
14,ESC-01,8.0,12.00,15.25


**O que observar:** Na primeira operação, **95 litros ÷ 8 horas = 11,875 L/h** e **120 m³ ÷ 8 horas = 15 m³/h**. A tabela arredonda apenas a exibição. A pergunta agora muda de “quanto produziu?” para “quanto produziu por hora?”.


## 26.5. Comparar os indicadores por equipamento

Primeiro somamos horas, combustível e produção de cada equipamento. Depois dividimos os totais. Assim, operações de durações diferentes contribuem de acordo com suas horas trabalhadas.


In [29]:
resumo = analise.groupby("equipamento")[["horas_trabalhadas", "combustivel_litros", "producao_m3"]].sum()

resumo["consumo_l_h"] = resumo["combustivel_litros"] / resumo["horas_trabalhadas"]
resumo["produtividade_m3_h"] = resumo["producao_m3"] / resumo["horas_trabalhadas"]

resumo.round(2)


,horas_trabalhadas,combustivel_litros,producao_m3,consumo_l_h,produtividade_m3_h
equipamento,,,,,
ESC-01,32.5,383.0,487,11.78,14.98
ESC-02,16.0,180.0,200,11.25,12.50
ESC-03,10.0,150.0,180,15.00,18.00
ESC-04,6.0,68.0,85,11.33,14.17
ESC-05,9.0,115.0,140,12.78,15.56


**O que observar:** ESC-01 acumula **32,5 horas**, consome aproximadamente **11,78 L/h** e produz **14,98 m³/h**. ESC-03 apresenta **15 L/h** e **18 m³/h**, em uma única operação. Consumo mais alto pode acompanhar produção mais alta. Esses valores, sozinhos, não comprovam defeito nem superioridade; tipo de máquina, tarefa e condições de trabalho também importam.


**Uma diferença importante, sem complicar:** para saber o consumo por hora do conjunto,
dividimos o combustível total pelas horas totais. Uma média simples dos consumos de cada operação
daria o mesmo peso a uma operação curta e a uma longa; ela responde a outra pergunta.

| Equipamento | Consumo do conjunto de operações (L/h) | Produtividade do conjunto de operações (m³/h) |
|---|---:|---:|
| ESC-01 | 11,78 | 14,98 |
| ESC-02 | 11,25 | 12,50 |
| ESC-03 | 15,00 | 18,00 |
| ESC-04 | 11,33 | 14,17 |
| ESC-05 | 12,78 | 15,56 |


## 26.6. Qual obra concentrou a produção?

Podemos usar a mesma ideia de agrupamento mudando somente a coluna que define os grupos.


In [30]:
producao_por_obra = analise.groupby("obra")["producao_m3"].sum()
producao_por_obra.sort_values(ascending=False)


obra
OBRA A    627
OBRA B    285
OBRA C    180
Name: producao_m3, dtype: int64

**O que observar:** **OBRA A: 627 m³; OBRA B: 285 m³; OBRA C: 180 m³.** A OBRA A concentra a maior produção observada. Isso não significa que a obra inteira esteja mais adiantada: não conhecemos suas metas nem todos os seus registros.


### 26.7. Escrever uma conclusão apoiada nos dados

Exemplo de conclusão:

> Nas nove operações analisadas, foram registradas 73,5 horas e 1.092 m³ de produção.
> ESC-01 acumulou o maior volume, com 487 m³ em quatro operações.
> ESC-03 apresentou a maior produtividade por hora observada, mas com apenas uma operação disponível.

**Agora responda com suas palavras:**

1. Qual equipamento mais produziu no total? Ele também teve a maior produtividade por hora?
2. Qual informação foi acrescentada quando dividimos produção por horas?
3. Qual comparação exige mais cuidado por ter poucas observações?
4. O que poderia mudar se conseguíssemos recuperar as seis operações pendentes?

Para continuar a atividade, escolha duas perguntas sobre sua base tratada, calcule as métricas
e escreva uma frase de interpretação para cada resultado. Informe quantos registros participaram de cada cálculo.

**O objetivo é conseguir explicar o número que calculamos.**
